In [ ]:
from snowflake.snowpark.context import get_active_session
import subprocess, sys
session = get_active_session()

for whl in ['python_pptx-1.0.2-py3-none-any.whl', 'XlsxWriter-3.2.2-py3-none-any.whl']:
    session.file.get(f'@AUDITED_FINANCIALS.COMMON.PPTX_STAGE/{whl}', '/tmp/')
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', f'/tmp/{whl}'], capture_output=True, text=True)
    print(result.stdout.strip())

import pptx
print(f'\npython-pptx {pptx.__version__} ready')

In [ ]:
import sys, os, importlib, shutil
sys.path.insert(0, '../clients/MEDSTAR_HEALTH/presentations')

import generate_bs_deep_dive as gen
importlib.reload(gen)

gen.OUT_PATH = '/tmp/deep_dive_balance_sheet.pptx'

try:
    gen.build_deck()
    dest = '../clients/MEDSTAR_HEALTH/presentations/deep_dive_balance_sheet.pptx'
    shutil.copy2('/tmp/deep_dive_balance_sheet.pptx', dest)
    print(f'PPTX saved to {dest}')
    print(f'File size: {os.path.getsize(dest):,} bytes')
except Exception as e:
    import traceback
    traceback.print_exc()

In [ ]:
from pptx import Presentation
import os

path = '/tmp/deep_dive_balance_sheet.pptx'
assert os.path.exists(path), f'PPTX not found at {path}'

prs = Presentation(path)
print(f'Slide count: {len(prs.slides)}')
print(f'Slide dimensions: {prs.slide_width/914400:.3f}" x {prs.slide_height/914400:.3f}"')
print()

for i, slide in enumerate(prs.slides, 1):
    shapes = list(slide.shapes)
    chart_count = sum(1 for s in shapes if s.has_chart)
    table_count = sum(1 for s in shapes if s.has_table)
    text_shapes = sum(1 for s in shapes if s.has_text_frame)

    title_text = ''
    for s in shapes:
        if s.has_text_frame:
            txt = s.text_frame.text.strip()
            if len(txt) > 20 and 'cokergroup' not in txt.lower() and 'pg.' not in txt.lower() and 'Source:' not in txt:
                title_text = txt[:80]
                break

    status = '\u2713' if (chart_count > 0 or table_count > 0 or i == 1) else '\u25cb'
    print(f'  Slide {i:2d}: {status}  shapes={len(shapes):2d}  charts={chart_count}  tables={table_count}  | {title_text}')

In [ ]:
session.file.put('/tmp/deep_dive_balance_sheet.pptx', '@AUDITED_FINANCIALS.COMMON.PPTX_STAGE', auto_compress=False, overwrite=True)
print('PPTX staged to @AUDITED_FINANCIALS.COMMON.PPTX_STAGE/deep_dive_balance_sheet.pptx')
print('Download with:')
print('  GET @AUDITED_FINANCIALS.COMMON.PPTX_STAGE/deep_dive_balance_sheet.pptx file:///tmp/')

In [ ]:
%%sql -r remove_result
REMOVE @AUDITED_FINANCIALS.COMMON.PPTX_STAGE/deep_dive_balance_sheet.pptx

In [ ]:
%%sql -r put_result
PUT 'file:///tmp/deep_dive_balance_sheet.pptx' @AUDITED_FINANCIALS.COMMON.PPTX_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE

In [ ]:
import os
print(f'File exists: {os.path.exists("/tmp/deep_dive_balance_sheet.pptx")}')
print(f'File size: {os.path.getsize("/tmp/deep_dive_balance_sheet.pptx"):,} bytes')

result = session.file.put(
    'file:///tmp/deep_dive_balance_sheet.pptx',
    '@AUDITED_FINANCIALS.COMMON.PPTX_STAGE',
    auto_compress=False,
    overwrite=True
)
for row in result:
    print(dict(row.as_dict()))

In [ ]:
import os, zipfile, hashlib

path = '/tmp/deep_dive_balance_sheet.pptx'

print(f'File size: {os.path.getsize(path):,} bytes')

try:
    with zipfile.ZipFile(path, 'r') as z:
        names = z.namelist()
        print(f'Valid ZIP: YES ({len(names)} entries)')
        has_content_types = '[Content_Types].xml' in names
        has_presentation = any('presentation.xml' in n for n in names)
        print(f'  [Content_Types].xml: {has_content_types}')
        print(f'  presentation.xml: {has_presentation}')
        print(f'  Slide count: {sum(1 for n in names if n.startswith("ppt/slides/slide") and n.endswith(".xml"))}')
except zipfile.BadZipFile as e:
    print(f'INVALID ZIP: {e}')

with open(path, 'rb') as f:
    header = f.read(4)
    print(f'File header bytes: {header.hex()} (PK=504b0304)')
    f.seek(0)
    md5 = hashlib.md5(f.read()).hexdigest()
    print(f'MD5: {md5}')

In [ ]:
import hashlib

session.sql("REMOVE @AUDITED_FINANCIALS.COMMON.PPTX_STAGE/deep_dive_balance_sheet.pptx").collect()

result = session.sql(
    "PUT 'file:///tmp/deep_dive_balance_sheet.pptx' @AUDITED_FINANCIALS.COMMON.PPTX_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
).collect()
for row in result:
    print(row)

stage_list = session.sql("LIST @AUDITED_FINANCIALS.COMMON.PPTX_STAGE PATTERN='.*balance_sheet.*'").collect()
for row in stage_list:
    print(f'Staged: {row[0]}, size={row[1]}, md5={row[2]}')

with open('/tmp/deep_dive_balance_sheet.pptx', 'rb') as f:
    local_md5 = hashlib.md5(f.read()).hexdigest()
    print(f'Local MD5:  {local_md5}')
    print(f'Match: {local_md5 == row[2]}')

In [ ]:
import hashlib

result = session.sql(
    "PUT 'file:///tmp/deep_dive_balance_sheet.pptx' @AUDITED_FINANCIALS.COMMON.PPTX_DOWNLOAD_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
).collect()
for row in result:
    print(row)

stage_list = session.sql("LIST @AUDITED_FINANCIALS.COMMON.PPTX_DOWNLOAD_STAGE").collect()
for row in stage_list:
    print(f'Staged: size={row[1]}')

with open('/tmp/deep_dive_balance_sheet.pptx', 'rb') as f:
    print(f'Local size: {len(f.read())}')

url = session.sql(
    "SELECT GET_PRESIGNED_URL(@AUDITED_FINANCIALS.COMMON.PPTX_DOWNLOAD_STAGE, 'deep_dive_balance_sheet.pptx', 86400) AS URL"
).collect()[0][0]
print(f'\nDownload URL:\n{url}')

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Emu

prs = Presentation('/tmp/deep_dive_balance_sheet.pptx')

for i, slide in enumerate(prs.slides, 1):
    shapes = list(slide.shapes)
    print(f'\n=== SLIDE {i} ({len(shapes)} shapes) ===')
    for s in shapes:
        desc = []
        if s.has_chart:
            chart = s.chart
            desc.append(f'CHART({chart.chart_type}, {len(chart.plots[0].series)} series)')
        if s.has_table:
            t = s.table
            desc.append(f'TABLE({t.rows.__len__()}r x {len(t.columns)}c)')
        if s.has_text_frame:
            txt = s.text_frame.text.strip()
            if txt:
                desc.append(f'TEXT: "{txt[:60]}"')
            else:
                desc.append('TEXT: (empty)')
        if not desc:
            desc.append(f'SHAPE: {s.shape_type}')
        w = s.width / 914400 if s.width else 0
        h = s.height / 914400 if s.height else 0
        print(f'  [{w:.1f}"x{h:.1f}"] {" | ".join(desc)}')

In [ ]:
from pptx import Presentation

prs = Presentation('/tmp/deep_dive_balance_sheet.pptx')

for i, slide in enumerate(prs.slides, 1):
    for s in slide.shapes:
        if s.has_chart:
            chart = s.chart
            ct = str(chart.chart_type)
            plot = chart.plots[0]
            n_series = len(plot.series)
            cats = [str(c) for c in chart.plots[0].categories] if hasattr(chart.plots[0], 'categories') else []
            print(f'Slide {i}: type={ct}, series={n_series}, categories={len(cats)}')
            for si, ser in enumerate(plot.series):
                vals = list(ser.values)
                print(f'  Series {si}: {vals[:4]}... ({len(vals)} pts)')
                has_fill = ser.format.fill.type is not None
                print(f'  Fill type: {ser.format.fill.type}')

In [ ]:
import importlib, sys
sys.path.insert(0, '../clients/MEDSTAR_HEALTH/presentations')
import generate_bs_deep_dive as gen
importlib.reload(gen)

gen.OUT_PATH = '/tmp/deep_dive_balance_sheet.pptx'
try:
    gen.build_deck()
    print('Build succeeded')
except Exception:
    import traceback
    traceback.print_exc()

import os
from pptx import Presentation as P
prs = P('/tmp/deep_dive_balance_sheet.pptx')
print(f'\nSlides: {len(prs.slides)}')
for i, slide in enumerate(prs.slides, 1):
    shapes = list(slide.shapes)
    charts = [s for s in shapes if s.has_chart]
    for c in charts:
        ch = c.chart
        n_plots = len(ch.plots)
        types = [str(ch.plots[j].chart_type) if hasattr(ch.plots[j],'chart_type') else '?' for j in range(n_plots)]
        print(f'  Slide {i}: {n_plots} plot(s) — {types}')

result = session.sql(
    "PUT 'file:///tmp/deep_dive_balance_sheet.pptx' @AUDITED_FINANCIALS.COMMON.PPTX_DOWNLOAD_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
).collect()
print(f'\nStaged: {result[0][3]} bytes')

url = session.sql(
    "SELECT GET_PRESIGNED_URL(@AUDITED_FINANCIALS.COMMON.PPTX_DOWNLOAD_STAGE, 'deep_dive_balance_sheet.pptx', 86400)"
).collect()[0][0]
print(f'\nDownload: {url}')